# FSDP 原理与 TorchTitan 框架

> 本 notebook 承接 02.01 的章节概览，深入讲解 FSDP 原理、TorchTitan 启动流程和 model_registry 机制。

---

## 1. 为什么需要 FSDP：1.7B 模型的显存困境

Qwen3-1.7B 有 17 亿参数，每个参数 bf16 格式占 2 字节：

$$\text{模型权重} = 1.7\text{B} \times 2 \text{ bytes} \approx 3.4 \text{ GiB}$$

训练时显存远不止权重：

| 组件 | 大小 | 说明 |
|------|------|------|
| 模型权重 | ~3.4 GiB | bf16，1 份 |
| 梯度 | ~3.4 GiB | 反向传播产生，1 份 |
| 优化器状态 | ~20 GiB | Adam 的 m 和 v，fp32，每个参数 4B×2 |
| 激活值 | ~10 GiB | 取决于 batch_size × seq_len |
| **合计** | **~37 GiB** | 远超单卡 32 GiB |

解决方案：把模型**切开来放**到多张卡上。这就是 FSDP 做的事。

---

## 2. FSDP 基本原理

FSDP（Fully Sharded Data Parallel）是 PyTorch 生态的分布式训练方案。核心思路：参数切成 N 份，每张卡只存 1/N。

### 2.1 正向传播：拼起来再算

![FSDP Forward](./images/fsdp-forward.png)

- 每张卡只存 1/N 参数（分片）
- 正向计算前，all-gather 拼出完整参数
- 各自用不同的 batch 数据前向传播
- 算完立即释放不用的参数片段

### 2.2 反向传播：reduce-scatter

![FSDP Backward](./images/fsdp-backward.png)

每张卡只算了自己 batch 的梯度——不完整。怎么做？

- **① reduce（求和）**：把所有 rank 的梯度加起来 → 完整梯度
- **② scatter（分发）**：按分片切回各卡，每卡只保留自己负责的那份

为什么不能直接 scatter？因为每张卡只算了自己的 batch，梯度不完整。必须先 reduce 得到「所有人的结果」，再 scatter。这就是 reduce-scatter。

### 2.3 FSDP2：通信藏进计算里

FSDP1 的问题：整个 layer 的 backward 做完才通信，通信时卡在等待。

FSDP2 的改进：把通信拆散到每一层。卡 0 在算第 3 层的梯度，卡 1 已经在传第 4 层的参数——**通信与计算 overlap**，通信时间几乎被「藏」起来了。

---

## 3. TorchTitan：FSDP 的开箱即用包装

FSDP 是 PyTorch 提供的底层能力，直接用需要写大量分布式代码。TorchTitan 把它包装成**一条命令启动**的训练框架。
大模型训练不只是「跑一个训练循环」，还需要：

- **分布式训练**：模型太大/输入太大/序列太长，一张卡放不下
- **数据加载**：对话数据的 tokenization、label masking、batch 拼接
- **检查点管理**：中断恢复、HuggingFace 格式权重导入导出
- **混合精度**：fp32/bf16 切换，省显存
- **优化器与学习率调度**：Adam 状态管理、warmup/decay
- **性能监控**：loss、grad_norm、吞吐量日志

TorchTitan 把这些能力模块化打包，开发者只需写一个配置文件。

### 3.1 启动流程

> 源码：`torchtitan_npu/entry.py`

![Training Startup Flow](./images/training-startup-flow.png)

### 3.2 训练脚本

> 源码：`scripts/run_train.sh`

```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh
```

| 参数 | 含义 |
|------|------|
| `NGPU=2` | 使用 2 张 NPU 卡 |
| `MODULE=torchtitan_npu.models.qwen3` | 使用 Qwen3 的 model_registry |
| `CONFIG=sft_qwen3_1_7b_wordle` | 使用 Wordle SFT 配置 |

---

## 4. model_registry：模型怎么被框架发现

> 源码：`torchtitan_npu/models/qwen3/__init__.py`

每个模型通过 `model_registry()` 注册自己：

```python
def model_registry(flavor: str):
    spec = _upstream_model_registry(flavor)
    return spec.__class__(
        name=spec.name,                              # 模型名称
        flavor=spec.flavor,                          # 规格（1.7B / 0.6B / debugmodel）
        model=spec.model,                            # 模型类（nn.Module）
        parallelize_fn=parallelize_qwen3,            # ← FSDP 分片逻辑挂载点
        pipelining_fn=spec.pipelining_fn,            # 流水线并行（本章不用）
        build_loss_fn=spec.build_loss_fn,            # loss 函数
        post_optimizer_build_fn=spec.post_optimizer_build_fn,
        state_dict_adapter=Qwen3StateDictAdapter,    # HF 格式权重加载
    )
```

| 字段 | 作用 |
|------|------|
| `model` | 模型结构定义（Qwen3 的 Transformer） |
| `parallelize_fn` | **FSDP 分片入口**——框架构建完模型后调用此函数，将参数切分到多卡 |
| `state_dict_adapter` | 加载/保存 HuggingFace 格式权重 |
| `flavor` | `"1.7B"` → 1.7B 参数版本，`"0.6B"` → 更小的调试版本 |
| `build_loss_fn` | 训练用的 loss 函数 |

配置中 `model_spec=model_registry("1.7B")` 一行，框架就知道用哪个模型、怎么分片、怎么加载权重。

---

## 小结

- **FSDP** 通过参数分片让多卡协作训练大模型——正向 all-gather 拼参数，反向 reduce-scatter 聚合梯度。
- **FSDP2** 把通信拆到每层，与计算 overlap，通信开销几乎不可见。
- **TorchTitan** 把 FSDP、数据处理、检查点、混合精度等能力打包成一条命令启动的训练框架。
- **model_registry** 注册模型，`parallelize_fn` 是 FSDP 分片的挂载点，`state_dict_adapter` 处理权重加载。

ModelConverter 机制将在第 4 章详细讲解——它允许你无侵入地替换模型中的算子。

## 练习

1. （判断题）FSDP 把模型参数切成 N 份，每张卡只存 1/N。正向时需要 all-gather 拼成完整模型再各自计算。

2. （单选题）反向传播时，FSDP 使用 reduce-scatter 而不是直接 scatter，原因是什么？
    A. scatter 速度太慢
    B. 每张卡只算了自己 batch 的梯度——不完整，必须先把所有 rank 的梯度求和（reduce），再分发（scatter）
    C. reduce-scatter 是 scatter 的别名
    D. scatter 已被 PyTorch 废弃

3. （判断题）FSDP2 相比 FSDP1 的核心改进是将通信拆散到每一层，与计算 overlap，通信时间几乎被「藏」起来。

4. （单选题）TorchTitan 的 `entry.py` 做了什么？
    A. 下载模型权重
    B. ConfigManager 解析参数 → 构建 Trainer → 启动训练循环
    C. 编译 CUDA kernel
    D. 启动推理服务

5. （单选题）`model_registry("1.7B")` 中 `parallelize_fn=parallelize_qwen3` 的作用是什么？
    A. 下载数据集
    B. 将模型参数按 FSDP 策略分片到多卡
    C. 转换模型为 ONNX 格式
    D. 启动 Web 服务器

In [ ]:
!cat ./answer/02.01_answer.txt